In [1]:
import pandas as pd
import numpy as np
import os

print("Environment ready.")

Environment ready.


In [2]:
print("Loading feature-engineered demand and clean weather datasets...")
df_hourly = pd.read_csv('../data/processed/hourly_zone_demand.csv')
df_weather = pd.read_csv('../data/processed/clean_weather.csv')

print("Hourly Zone Demand Records:", len(df_hourly))
print("Clean Weather Records:", len(df_weather))

Loading feature-engineered demand and clean weather datasets...


Hourly Zone Demand Records: 11994624
Clean Weather Records: 1096


In [3]:
print("Integrating datasets and finalising feature engineering...")
# 1. Ensure date columns are formatted as datetime types for exact matching
df_hourly['date'] = pd.to_datetime(df_hourly['date'])
df_weather['date'] = pd.to_datetime(df_weather['date'])

# 2. Perform a high-speed left join to attach daily weather conditions to each hourly zone row
df_combined = pd.merge(df_hourly, df_weather, on='date', how='left')

# 3. Engineer final weather features for the ML model
# Fill missing precipitation or rain with 0.0 (if any)
df_combined['rain'] = df_combined['rain'].fillna(0.0)
df_combined['is_raining'] = (df_combined['rain'] > 0.0).astype(int)

print("Final combined dataset shape:", df_combined.shape)
df_combined.head(10)

Integrating datasets and finalising feature engineering...


Final combined dataset shape: (11994624, 12)


,zone_id,date,daytype,hour,demand,is_weekend,temp_max,temp_min,precipitation,rain,weather_code,is_raining
0,ZONE_41.6850_-87.6100,2019-01-01,U,0,1.38,1,2.3,-1.5,6.0,3.6,73,1
1,ZONE_41.6850_-87.6100,2019-01-01,U,1,0.69,1,2.3,-1.5,6.0,3.6,73,1
2,ZONE_41.6850_-87.6100,2019-01-01,U,2,0.28,1,2.3,-1.5,6.0,3.6,73,1
3,ZONE_41.6850_-87.6100,2019-01-01,U,3,0.28,1,2.3,-1.5,6.0,3.6,73,1
4,ZONE_41.6850_-87.6100,2019-01-01,U,4,0.69,1,2.3,-1.5,6.0,3.6,73,1
5,ZONE_41.6850_-87.6100,2019-01-01,U,5,1.38,1,2.3,-1.5,6.0,3.6,73,1
6,ZONE_41.6850_-87.6100,2019-01-01,U,6,2.76,1,2.3,-1.5,6.0,3.6,73,1
7,ZONE_41.6850_-87.6100,2019-01-01,U,7,5.52,1,2.3,-1.5,6.0,3.6,73,1
8,ZONE_41.6850_-87.6100,2019-01-01,U,8,8.28,1,2.3,-1.5,6.0,3.6,73,1
9,ZONE_41.6850_-87.6100,2019-01-01,U,9,9.65,1,2.3,-1.5,6.0,3.6,73,1


In [4]:
print("Writing final integrated dataset (combined_dataset.csv) to disk...")
df_combined.to_csv('../data/processed/combined_dataset.csv', index=False)
print("combined_dataset.csv successfully generated in /data/processed/!")

Writing final integrated dataset (combined_dataset.csv) to disk...


combined_dataset.csv successfully generated in /data/processed/!


In [5]:
import pandas as pd

df = pd.read_csv('../data/processed/combined_dataset.csv')

print("=== SHAPE ===")
print(df.shape)

print("\n=== NULL VALUES ===")
print(df.isnull().sum())

print("\n=== BASIC STATS ===")
print(df.describe())

print("\n=== UNIQUE DAYTYPES ===")
print(df['daytype'].unique())

print("\n=== DEMAND RANGE ===")
print("Min demand:", df['demand'].min())
print("Max demand:", df['demand'].max())
print("Average demand:", df['demand'].mean())

print("\n=== DATE RANGE ===")
print("From:", df['date'].min())
print("To:", df['date'].max())

print("\n=== ZONE COUNT ===")
print("Total zones:", df['zone_id'].nunique())

=== SHAPE ===
(11994624, 12)

=== NULL VALUES ===
zone_id          0
date             0
daytype          0
hour             0
demand           0
is_weekend       0
temp_max         0
temp_min         0
precipitation    0
rain             0
weather_code     0
is_raining       0
dtype: int64

=== BASIC STATS ===
               hour        demand    is_weekend      temp_max      temp_min  \
count  1.199462e+07  1.199462e+07  1.199462e+07  1.199462e+07  1.199462e+07   
mean   1.150000e+01  9.795386e+00  3.020073e-01  1.494279e+01  6.917336e+00   
std    6.922187e+00  1.204554e+01  4.591284e-01  1.090923e+01  1.045432e+01   
min    0.000000e+00  4.000000e-02  0.000000e+00 -2.000000e+01 -3.250000e+01   
25%    5.750000e+00  1.390000e+00  0.000000e+00  5.900000e+00 -9.000000e-01   
50%    1.150000e+01  6.160000e+00  0.000000e+00  1.490000e+01  6.200000e+00   
75%    1.725000e+01  1.359000e+01  1.000000e+00  2.490000e+01  1.682500e+01   
max    2.300000e+01  1.823400e+02  1.000000e+00  3.53000

In [6]:
print("Rows per zone:", df.groupby('zone_id').size().mean())
print("Sample zones:", df['zone_id'].unique()[:5])

Rows per zone: 26304.0
Sample zones: ['ZONE_41.6850_-87.6100' 'ZONE_41.6900_-87.6100' 'ZONE_41.6900_-87.6150'
 'ZONE_41.6950_-87.6100' 'ZONE_41.6950_-87.6150']


In [7]:
top_zones = df.groupby('zone_id')['demand'].mean().sort_values(ascending=False).head(50).index

df_small = df[df['zone_id'].isin(top_zones)]

print("Reduced rows:", df_small.shape)
print("Zones kept:", df_small['zone_id'].nunique())

df_small.to_csv('../data/processed/combined_dataset_small.csv', index=False)
print("Saved successfully")

Reduced rows: (1315200, 12)
Zones kept: 50
Saved successfully
